# FEVS 2024 – Step 3: Build the indices

**Goal:** score engagement and well-being, first for all federal employees, then for each agency and work unit.

1. Rebuild OPM's four official indices and check them against OPM's published numbers
2. Design my own **Well-being & Workload Index** and test that its questions belong together
3. Score every agency and work unit
4. Give every respondent a personal score, for the driver analysis in Step 4

**OPM's method** (from the *Indices and Dimensions Guide*): for each question, take the weighted
**% positive** (answers 4 or 5). An index = the average of its questions' % positive.
The Engagement Index = the average of its three sub-indices.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

CLEAN = Path("../data/clean")
df = pd.read_csv(CLEAN / "fevs_2024_clean.csv.gz", low_memory=False)
questions = pd.read_csv(CLEAN / "fevs_2024_questions.csv").set_index("question")
print(f"Loaded {len(df):,} respondents")

Loaded 674,207 respondents


## 1. Which questions belong to which index

OPM's four indices come from the codebook. The **Well-being & Workload Index** is my own design:
it combines the questions about workload and work-life balance.

In [2]:
INDICES = {
    # OPM Employee Engagement Index – three sub-indices
    "EEI – Leaders lead":             ["Q57", "Q58", "Q59", "Q61", "Q62"],
    "EEI – Supervisors":              ["Q48", "Q50", "Q51", "Q52", "Q54"],
    "EEI – Intrinsic work experience":["Q2", "Q3", "Q4", "Q6", "Q7"],
    # Other OPM indices
    "Global Satisfaction":            ["Q46", "Q70", "Q71", "Q72"],
    "Performance Confidence":         ["Q20", "Q21", "Q22", "Q23"],
    "Employee Experience":            ["Q86", "Q87", "Q88", "Q89", "Q90"],
    # My own index
    "Well-being & Workload":          ["Q5", "Q34", "Q49", "Q63"],
}
EEI_SUBS = ["EEI – Leaders lead", "EEI – Supervisors", "EEI – Intrinsic work experience"]

(pd.DataFrame([(name, q, questions.loc[q, "text"]) for name, qs in INDICES.items() for q in qs],
              columns=["index", "question", "text"])
   .query("index == 'Well-being & Workload'"))

,index,question,text
28,Well-being & Workload,Q5,My workload is reasonable.
29,Well-being & Workload,Q34,Employees in my work unit support my need to b...
30,Well-being & Workload,Q49,My supervisor supports my need to balance work...
31,Well-being & Workload,Q63,Senior leaders demonstrate support for Work-Li...


## 2. The scoring function

`pct_positive` gives the weighted % of people who answered 4 or 5, ignoring blanks.
It works for everyone at once, or for every group (agency, work unit) in one go.

In [3]:
def pct_positive(data, items, by=None):
    """Weighted % positive (4–5) for each item; one row per group if `by` is given."""
    w = data["POSTWT"]
    answered = data[items].notna().mul(w, axis=0)          # weight of people who answered
    positive = (data[items] >= 4).mul(w, axis=0)             # weight of people who agreed
    if by is None:
        return 100 * positive.sum() / answered.sum()
    return 100 * positive.groupby(data[by]).sum() / answered.groupby(data[by]).sum()

def score_indices(data, by=None):
    """Average the items' % positive into each index (OPM method)."""
    out = {}
    for name, items in INDICES.items():
        item_scores = pct_positive(data, items, by)
        out[name] = item_scores.mean(axis=1) if by else item_scores.mean()
    out = pd.DataFrame(out) if by else pd.Series(out)
    eei = out[EEI_SUBS].mean(axis=1) if by else out[EEI_SUBS].mean()
    if by:
        out.insert(0, "Employee Engagement (EEI)", eei)
    else:
        out = pd.concat([pd.Series({"Employee Engagement (EEI)": eei}), out])
    return out

## 3. Governmentwide scores – do we match OPM?

OPM published an **Engagement Index of 73** and an **Employee Experience Index of 74** for 2024.
If our numbers are close, the method is right.

In [4]:
gov = score_indices(df).round(1).rename("score").to_frame()
gov["OPM published"] = pd.Series({"Employee Engagement (EEI)": 73, "Employee Experience": 74})
gov

,score,OPM published
Employee Engagement (EEI),73.0,73.0
EEI – Leaders lead,62.7,NaN
EEI – Supervisors,81.3,NaN
EEI – Intrinsic work experience,74.9,NaN
Global Satisfaction,65.4,NaN
Performance Confidence,84.3,NaN
Employee Experience,74.4,74.0
Well-being & Workload,71.3,NaN


## 4. Is my Well-being & Workload Index a fair index?

Two checks:
- **Cronbach's alpha** – do the four questions measure the same thing? (0.7 or higher is usually seen as acceptable)
- **Item-to-rest correlation** – does each question fit with the other three?

In [5]:
def cronbach_alpha(items_df):
    d = items_df.dropna()
    k = d.shape[1]
    return k / (k - 1) * (1 - d.var(ddof=1).sum() / d.sum(axis=1).var(ddof=1))

wb_items = INDICES["Well-being & Workload"]
wb = df[wb_items].dropna()
print(f"Cronbach's alpha: {cronbach_alpha(df[wb_items]):.2f}  (based on {len(wb):,} complete answers)")

fit = pd.Series({q: wb[q].corr(wb.drop(columns=q).mean(axis=1)) for q in wb_items}, name="item-rest r").round(2)
fit.to_frame().join(questions["text"])

Cronbach's alpha: 0.78  (based on 599,966 complete answers)


,item-rest r,text
Q5,0.49,My workload is reasonable.
Q34,0.69,Employees in my work unit support my need to b...
Q49,0.63,My supervisor supports my need to balance work...
Q63,0.58,Senior leaders demonstrate support for Work-Li...


## 5. Scores for every agency and work unit

We also count how many people answered, so small groups can be spotted later.

In [6]:
agency = score_indices(df, by="agency_name")
agency.insert(0, "respondents", df.groupby("agency_name").size())
agency = agency.round(1).sort_values("Employee Engagement (EEI)", ascending=False)

unit = score_indices(df, by="level1")   # by code: some work units share a name across agencies
unit.insert(0, "respondents", df.groupby("level1").size())
unit.insert(0, "level1_name", df.groupby("level1")["level1_name"].first())
unit.insert(0, "agency_name", df.groupby("level1")["agency_name"].first())
unit = unit.round(1).sort_values("Employee Engagement (EEI)", ascending=False)

print(f"{len(agency)} agencies, {len(unit)} work units scored")
cols = ["respondents", "Employee Engagement (EEI)", "Well-being & Workload", "Global Satisfaction"]
print("\nTop 5 agencies by engagement");   display(agency[cols].head(5))
print("Bottom 5 agencies by engagement"); display(agency[cols].tail(5))

36 agencies, 163 work units scored

Top 5 agencies by engagement


,respondents,Employee Engagement (EEI),Well-being & Workload,Global Satisfaction
agency_name,,,,
Pension Benefit Guaranty Corporation,621,89.0,91.9,87.4
Federal Energy Regulatory Commission,1201,85.7,85.9,79.9
General Services Administration,8268,85.6,85.0,82.1
Office of Management and Budget,500,81.9,79.0,77.6
Environmental Protection Agency,9366,80.7,78.3,77.3


Bottom 5 agencies by engagement


,respondents,Employee Engagement (EEI),Well-being & Workload,Global Satisfaction
agency_name,,,,
U.S. Agency for Global Media,561,69.9,70.1,65.7
U.S. Agency for International Development,1731,69.9,65.6,63.1
Department of Homeland Security,94177,68.8,66.6,64.4
Social Security Administration,40016,68.4,59.0,52.7
Department of Justice,38897,67.6,66.6,58.6


## 6. Does well-being move with engagement?

A first look: across agencies, do places with higher well-being also have higher engagement?

In [7]:
r = agency["Well-being & Workload"].corr(agency["Employee Engagement (EEI)"])
print(f"Correlation across {len(agency)} agencies: r = {r:.2f}")

Correlation across 36 agencies: r = 0.93


## 7. A personal score for every respondent

For Step 4 (what drives engagement) we need one score per person.
Each person's score = the **average of their answers (1–5)** on the index questions,
kept only if they answered at least half of them.

In [8]:
person = df[["RandomID", "POSTWT", "agency", "agency_name", "level1", "level1_name",
             "DSUPER", "DFEDTEN", "DAGEGRP", "DMIL", "DLEAVING", "plans_to_leave"]].copy()

def person_score(items):
    answers = df[items]
    enough = answers.notna().sum(axis=1) >= len(items) / 2
    return answers.mean(axis=1).where(enough).round(3)

person["eei_score"] = person_score(sum((INDICES[s] for s in EEI_SUBS), []))
for name, col in [("Global Satisfaction", "satisfaction_score"),
                  ("Employee Experience", "experience_score"),
                  ("Well-being & Workload", "wellbeing_score")]:
    person[col] = person_score(INDICES[name])

person[["eei_score", "satisfaction_score", "experience_score", "wellbeing_score"]].describe().round(2)

,eei_score,satisfaction_score,experience_score,wellbeing_score
count,659562.00,650755.00,648379.00,667324.00
mean,3.95,3.72,3.98,3.88
std,0.82,0.95,0.84,0.87
min,1.00,1.00,1.00,1.00
25%,3.53,3.25,3.60,3.50
50%,4.07,4.00,4.00,4.00
75%,4.60,4.50,4.80,4.50
max,5.00,5.00,5.00,5.00


## 8. Save everything for the next steps and the dashboard

In [9]:
gov.to_csv(CLEAN / "indices_governmentwide.csv")
agency.to_csv(CLEAN / "indices_by_agency.csv")
unit.to_csv(CLEAN / "indices_by_workunit.csv")
person.to_csv(CLEAN / "fevs_2024_person_scores.csv.gz", index=False, compression="gzip")
pd.DataFrame([(n, q) for n, qs in INDICES.items() for q in qs],
             columns=["index", "question"]).to_csv(CLEAN / "index_questions.csv", index=False)

for f in ["indices_governmentwide.csv", "indices_by_agency.csv", "indices_by_workunit.csv",
          "fevs_2024_person_scores.csv.gz", "index_questions.csv"]:
    print("Saved", CLEAN / f)

Saved ..\data\clean\indices_governmentwide.csv
Saved ..\data\clean\indices_by_agency.csv
Saved ..\data\clean\indices_by_workunit.csv
Saved ..\data\clean\fevs_2024_person_scores.csv.gz
Saved ..\data\clean\index_questions.csv


## Summary

- OPM's indices rebuilt and checked against the published 2024 numbers
- My Well-being & Workload Index designed and tested for consistency
- Scores saved for 36 agencies and 163 work units (ready for Power BI)
- One score per respondent saved for the driver analysis

**Next step (Step 4):** which workplace factors are most linked to engagement and to people planning to leave.